# Homework 6 - Decision Trees and Ensemble Learning


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb


### Data preparation and split


In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)

# Fill missing values with zeros
df = df.fillna(0)

# Split: 60% train, 20% validation, 20% test
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

feature_names = dv.get_feature_names_out()


## Question 1

Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable with `max_depth=1`.

Which original feature is used for splitting the data?


In [3]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

split_feature = feature_names[dt.tree_.feature[0]]
original_feature = split_feature.split('=')[0]

print(f"Split feature: '{original_feature}'")
print()
print(export_text(dt, feature_names=list(feature_names)))


Split feature: 'model_year'

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



## Question 2

Train a random forest regressor with `n_estimators=10`, `random_state=1`, and `n_jobs=-1`.

What's the RMSE of this model on the validation data?


In [4]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = mean_squared_error(y_val, y_pred) ** 0.5

print(f"Validation RMSE: {rmse:.4f}")


Validation RMSE: 1.8332


## Question 3

Now let's experiment with the `n_estimators` parameter in `[10, 50, 100, 150]` with `random_state=1`.

Which value gives the lowest validation RMSE? (Consider 3 decimal places).


In [5]:
scores = {}

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = mean_squared_error(y_val, y_pred) ** 0.5
    scores[n] = rmse
    print(f"n_estimators={n:3d}: RMSE = {rmse:.4f} (rounded: {round(rmse, 3)})")

best_n = min(scores, key=scores.get)
print(f"\nLowest validation RMSE with n_estimators={best_n}")


n_estimators= 10: RMSE = 1.8332 (rounded: 1.833)


n_estimators= 50: RMSE = 1.7740 (rounded: 1.774)


n_estimators=100: RMSE = 1.7698 (rounded: 1.77)


n_estimators=150: RMSE = 1.7692 (rounded: 1.769)

Lowest validation RMSE with n_estimators=150


## Question 4

Let's select the best `max_depth` from `[10, 15, 20, 25]` across `n_estimators` in `[10, 50, 100, 150]` with `random_state=1`.

What's the best `max_depth`, using the mean RMSE?


In [6]:
depth_means = {}

for depth in [10, 15, 20, 25]:
    rmse_list = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred) ** 0.5
        rmse_list.append(rmse)
    mean_rmse = np.mean(rmse_list)
    depth_means[depth] = mean_rmse
    print(f"max_depth={depth:2d}: mean RMSE = {mean_rmse:.4f}")

best_depth = min(depth_means, key=depth_means.get)
print(f"\nBest max_depth: {best_depth}")


max_depth=10: mean RMSE = 1.7544


max_depth=15: mean RMSE = 1.7842


max_depth=20: mean RMSE = 1.7868


max_depth=25: mean RMSE = 1.7863

Best max_depth: 10


## Question 5

Train `RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)` and inspect feature importances.

What's the most important feature (among vehicle_weight, horsepower, acceleration, engine_displacement)?


In [7]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importances = dict(zip(feature_names, rf.feature_importances_))
candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']

for feat in candidates:
    print(f"{feat:20s}: {importances.get(feat, 0.0):.4f}")

most_important = max(candidates, key=lambda f: importances.get(f, 0.0))
print(f"\nMost important feature: {most_important}")


vehicle_weight      : 0.2033
horsepower          : 0.0783
acceleration        : 0.0550
engine_displacement : 0.0613

Most important feature: vehicle_weight


## Question 6

Now let's train an XGBoost model for 100 rounds with `eta=0.3` and `eta=0.1`.

Which eta leads to the best RMSE score on the validation dataset?


In [8]:
clean_features = [f.replace('[', '_').replace(']', '_').replace('<', '_') for f in feature_names]

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=clean_features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=clean_features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 0,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    y_pred = model.predict(dval)
    rmse = mean_squared_error(y_val, y_pred) ** 0.5
    print(f"eta={eta}: validation RMSE = {rmse:.4f}")


eta=0.3: validation RMSE = 1.8265


eta=0.1: validation RMSE = 1.7248
